In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection  import TimeSeriesSplit
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    log_loss,
    classification_report
)

import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
sys.path.append(str(project_root))

from models.trees.features import ml_ready
from db.sqlite.queries_ohlcv import fetch_ohlcv
from db.utils_ohlcv import get_ibex_tickers, get_macro_tickers
from config import ARTIFACTS_PATH

In [ ]:
tickers = get_ibex_tickers()
df_micro = fetch_ohlcv(tickers)

t_macro = get_macro_tickers()
df_macro = fetch_ohlcv(t_macro)

df_micro = df_micro[df_micro["volume"] > 0]
# preprocess macro too (ibex vix no volume)
horizon = 1

df, X, y, mask= ml_ready(horizon, df_micro, df_macro, "macro")

dates = df.loc[mask,"date"]
unique_dates = np.sort(dates.unique())
tscv = TimeSeriesSplit(n_splits=5)
train_date_idx, test_date_idx = list(tscv.split(unique_dates))[-1]

train_dates = unique_dates[train_date_idx]
test_dates = unique_dates[test_date_idx]


train_mask = dates.isin(train_dates)
test_mask = dates.isin(test_dates)

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

model = XGBClassifier()

model.fit(X_train, y_train)

preds = model.predict(X_test)
proba = model.predict_proba(X_test)[:,1]


acc = accuracy_score(y_test, preds)
bal_acc = balanced_accuracy_score(y_test, preds)
roc = roc_auc_score(y_test, proba)
ll = log_loss(y_test, proba)
cd = y.value_counts(normalize=True)
mpc = proba.mean()
importances = (pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False))
print("\n===== VALIDATION METRICS =====")
print("Accuracy:", acc)
print("Balanced Accuracy:", bal_acc)
print("ROC AUC:", roc)
print("Log Loss:", ll)
print("Class distribution:", cd)
print("Mean predicted probability:", mpc)
print("\nClassification Report:")
print(classification_report(y_test, preds))
print("\nTop 10 Features:", importances.head(10))


metadata = {
    "features": list(X.columns),
    "n_features": X.shape[1],
    "train_start": str(train_dates.min()),
    "train_end": str(train_dates.max()),
    "test_start": str(test_dates.min()),
    "test_end": str(test_dates.max()),
    "train_size": len(X_train),
    "test_size": len(X_test),
    "accuracy": acc,
    "balanced_accuracy": bal_acc,
    "roc_auc": roc,
    "log_loss": ll,
    "mean_predicted_prob":mpc,
    "class_distribution":cd,
    "params": model.get_params(),
    "random_state": 42,
    "top_features": importances.head(10).to_dict(),
}

"""
joblib.dump(
    {
        "model": model,
        "metadata": metadata,
    },
    "rf_ibex_h1_validated.pkl"
)
"""


model.fit(X, y)

joblib.dump(
    {
        "model": model,
        "features": list(X.columns),
        "params": model.get_params(),
        "validation_metadata": metadata,
    },
    ARTIFACTS_PATH/ f"rf_h{horizon}_full.pkl"
)

print("\nFull model retrained and saved.")

 